# 第8周 · 作业（预计约6小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 从零写训练循环：`train_one_epoch` 与 `evaluate` | 2.5 小时 |
| B | MNIST 三方对比：MLP vs 逻辑回归 vs 随机森林 × 3 种子 | 2 小时 |
| C | Capstone：bit flip 工作表（3 个候选问题） | 1.5 小时 |

**规则**：本周作业不使用 AI 编程助手写代码；可以让 AI 解释报错信息。（这是禁令的最后一周——第 9 周起 AI 助手开放，但会有新的配套要求。）

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from zero2researcher import data, experiments
from zero2researcher.checks import week08 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 从零写训练循环

**不翻课件**，凭记忆写出这两个函数（写不出来再回去看，看完合上再写——这是本课最重要的一次肌肉记忆训练）。要求：

- `train_one_epoch(model, loader, criterion, optimizer, device)`：训练一个 epoch，返回**平均训练损失**（float）。开头 `model.train()`；每个 batch 先把数据搬到 device，然后执行五步核心；
- `evaluate(model, loader, device)`：返回准确率（float）。开头 `model.eval()`，整个循环包在 `torch.no_grad()` 里，**绝不能改参数**。

自查会在一份小合成数据上实际运行它们：检查 backward 之前梯度是否清零、损失是否下降、准确率算得对不对、评估有没有关梯度、评估之后再训练时模式是否切换回来。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回平均训练损失（float）。"""
    # 在这里写你的代码
    raise NotImplementedError


def evaluate(model, loader, device):
    """返回准确率（float）；eval 模式 + 全程 no_grad，绝不改模型。"""
    # 在这里写你的代码
    raise NotImplementedError


ck.train_loop(train_one_epoch, evaluate)

## B · MNIST 三方对比 × 3 个种子

重做课上的三方对比，但按第 6 周"最小实验单元"的标准升级：

- **训练集**：MNIST **前 10,000 张**（固定索引，三位选手完全一致）；测试集全量 10,000 张；
- **三位选手**：MLP（784-256-10，Adam 1e-3，3 epochs，batch 64，`Normalize((0.1307,), (0.3081,))`）、`LogisticRegression(max_iter=1000)`、`RandomForestClassifier(n_estimators=100, random_state=seed)`。输入：MLP 用 Normalize 后的图，LR/RF 用展平像素 / 255；
- **种子** 0、1、2：每轮开头调用 `experiments.set_seed(seed)`，并把种子传给随机森林的 `random_state`；
- **报告**：每位选手用 `experiments.format_result(accs)` 打印均值±标准差，并画一张带误差棒的图。

先运行数据准备（已给出）：

In [ ]:
def load_dataset(cls, train, transform=None):
    """下载并缓存数据集；网络抖动时自动重试一次。"""
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True, transform=transform)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = load_dataset(datasets.MNIST, train=True, transform=tfm)
test_set = load_dataset(datasets.MNIST, train=False, transform=tfm)

train_loader = DataLoader(Subset(train_full, range(10000)), batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=512)

X_train = train_full.data[:10000].reshape(10000, -1).numpy() / 255.0   # LR/RF 用：同样的 1 万张
y_train = train_full.targets[:10000].numpy()
X_test = test_set.data.reshape(-1, 784).numpy() / 255.0
y_test = test_set.targets.numpy()


def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

accs = {"MLP": [], "LR": [], "RF": []}
# 在这里写你的代码
raise NotImplementedError
for name_, vals_ in accs.items():
    print(f"{name_}: {experiments.format_result(vals_)}")
ck.mnist_mlp_accuracy(float(np.mean(accs["MLP"])))
ck.mnist_lr_accuracy(float(np.mean(accs["LR"])))
ck.mnist_rf_accuracy(float(np.mean(accs["RF"])))

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**问题**：逻辑回归三个种子的标准差是多少？为什么是这个值（提示：它的训练过程里有随机成分吗）？随机森林与 MLP 的差距和它们各自的标准差相比如何——按第 6 周的三分法（显著更好 / 显著更差 / 差异小于噪声），你的结论是什么？（三四句话）

（在这里回答）

## C · Capstone：bit flip 工作表

用课上的格式写 **3 个候选研究问题**（至少 1 个基于公开数据集；鼓励有 1 个来自你实验室的数据，但公开数据的备选必须保留）。每个候选写四行：

- **默认假设（bit）**：你的领域/任务里大家不加论证就接受的一句话；
- **翻转（flip）**：把它反过来的主张；
- **可检验预测（prediction）**：如果翻转成立，你能用什么数据、什么对比、看到什么**数字**；
- **自评**：数据可得？Colab 可跑？基线是什么？

示例（课上讲的那个）：

> - 默认假设：神经网络做图像分类理所当然优于经典机器学习。
> - 翻转：小数据、简单任务上随机森林不输小 MLP。
> - 可检验预测：MNIST 10k 子集上 RF ≥ 未精调的 MLP；数据缩小 5 倍后差距扩大。
> - 自评：数据可得 ✓（torchvision 一行）；Colab 可跑 ✓（分钟级）；基线 = MLP 与 RF 互为基线。

第 10 周你要从三个候选里挑一个写成一页纸提案，所以认真写——这不是填表，是你 Capstone 的第一稿。

### 候选 1

- 默认假设（bit）：
- 翻转（flip）：
- 可检验预测（prediction）：
- 自评（数据 / Colab / 基线）：

### 候选 2

- 默认假设（bit）：
- 翻转（flip）：
- 可检验预测（prediction）：
- 自评（数据 / Colab / 基线）：

### 候选 3

- 默认假设（bit）：
- 翻转（flip）：
- 可检验预测（prediction）：
- 自评（数据 / Colab / 基线）：

## 反思（2–3句话）

第 7 周手写 micrograd 的哪段经验让这周的 PyTorch 学起来更顺？有没有哪一步你觉得"PyTorch 在替我做我还不理解的事"？

（在这里写）